# Food to Recipe Generation using CLIP + MiniMax-M1-40k

## Project Overview

This notebook implements a deep learning system that generates cooking recipes from images of food using **CLIP** for food recognition and **MiniMax-M1-40k** for recipe generation.

### Key Features:
- ✅ **Vision Understanding**: CLIP for accurate food recognition
- ✅ **Large-Scale Hybrid-Attention Model**: MiniMax-M1-40k (456B params, 40K thinking tokens) for recipe generation
- ✅ **Better Generation Quality**: More coherent and accurate recipes with extended reasoning
- ✅ **Instruction Following**: MiniMax-M1-40k is trained with reinforcement learning for better prompts

### Approach
1. **CLIP**: Food recognition (100+ categories)
2. **MiniMax-M1-40k**: Large-scale hybrid-attention reasoning model for recipe generation
3. **Training**: Fine-tune MiniMax-M1-40k on food-recipe pairs
4. **Evaluation**: Monitor quality and coherence

### Model References:
- [MiniMax-M1-40k on Hugging Face](https://huggingface.co/MiniMaxAI/MiniMax-M1-40k)
- [MiniMax-M1 GitHub](https://github.com/MiniMax-AI/MiniMax-M1)


In [ ]:
# import sys
# import subprocess

# packages = [
#     "torch",
#     "torchvision", 
#     "transformers>=4.37.0",
#     "pillow",
#     "pandas",
#     "matplotlib",
#     "seaborn",
#     "accelerate",
#     "bitsandbytes",
#     "gradio>=5.0.0",
#     "einops"  # Required for MiniMax-M1-40k
# ]

# print("Installing required packages...")
# for package in packages:
#     print(f"Installing {package}...")
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

# print("\n✓ All packages installed successfully!")


## Installation Requirements

```bash
pip install torch torchvision
pip install transformers>=4.37.0
pip install pillow pandas matplotlib seaborn
pip install accelerate bitsandbytes
pip install gradio
pip install einops  # Required for MiniMax-M1-40k
```


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import zipfile
import os
import re
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Hugging Face Libraries
from transformers import (
    CLIPProcessor, CLIPModel,
    AutoModelForCausalLM,  # Use AutoModelForCausalLM for MiniMax-M1-40k
    AutoProcessor,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Set device - MULTI-GPU SUPPORT
# First, clear any existing CUDA state to avoid conflicts
if torch.cuda.is_available():
    try:
        torch.cuda.empty_cache()
        # Synchronize to catch any existing errors
        for i in range(torch.cuda.device_count()):
            try:
                torch.cuda.synchronize(i)
            except:
                pass
    except:
        pass

if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"Found {num_gpus} GPU(s)")
    
    if num_gpus > 1:
        device = torch.device("cuda")
        use_multi_gpu = True
        print(f"Using MULTI-GPU training with {num_gpus} GPUs")
        for i in range(num_gpus):
            gpu_name = torch.cuda.get_device_name(i)
            print(f"  GPU {i}: {gpu_name}")
    else:
        device = torch.device("cuda")
        use_multi_gpu = False
        print(f"Using single GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    use_multi_gpu = False
    print("Using CPU (No GPU available)")

print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")

print("\n✓ Libraries imported successfully!")
print("\n⚠️ If you encounter CUDA errors, try:")
print("   1. Restart the kernel")
print("   2. Run this cell again to reset CUDA state")
print("   3. If errors persist, restart Jupyter and clear GPU memory")


Found 2 GPU(s)
Using MULTI-GPU training with 2 GPUs
  GPU 0: NVIDIA GeForce RTX 3090
  GPU 1: NVIDIA GeForce RTX 3090

PyTorch version: 2.9.0+cu128
CUDA available: True
CUDA version: 12.8

✓ Libraries imported successfully!

⚠️ If you encounter CUDA errors, try:
   1. Restart the kernel
   2. Run this cell again to reset CUDA state
   3. If errors persist, restart Jupyter and clear GPU memory


## 1. Dataset Loading

Using the same dataset for consistency with other model versions.


In [3]:
# Extract dataset (reusing from original notebook)
zip_files = [
    "Food Ingredients and Recipe Dataset with Image Name Mapping.csv.zip"
]

for zip_file in zip_files:
    if os.path.exists(zip_file):
        print(f"Extracting {zip_file}...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall(".")
        print(f"✓ Extracted successfully!")

target_csv = "Food Ingredients and Recipe Dataset with Image Name Mapping.csv"
if os.path.exists(target_csv):
    print(f"\n✓ Found dataset: {target_csv}")
    main_dataset = pd.read_csv(target_csv)
    print(f"✓ Loaded {len(main_dataset)} recipes")
else:
    print(f"\n✗ Dataset '{target_csv}' not found!")



✓ Found dataset: Food Ingredients and Recipe Dataset with Image Name Mapping.csv
✓ Loaded 13501 recipes


## 2. Load CLIP for Food Recognition

CLIP is used for initial food recognition, which helps guide the MiniMax-M1-40k model in generating more accurate recipes.


In [4]:
# Load CLIP for food recognition (same as before)
print("Loading CLIP model (ViT-B/32)...")

# Reset CUDA state before loading CLIP to avoid conflicts
if torch.cuda.is_available():
    try:
        # Clear any existing CUDA errors
        torch.cuda.empty_cache()
        # Synchronize all CUDA operations
        for i in range(torch.cuda.device_count()):
            try:
                torch.cuda.synchronize(i)
            except:
                pass
        # Reset error state
        torch.cuda.reset_peak_memory_stats()
        print("✓ CUDA state cleared")
    except Exception as e:
        print(f"⚠️ Warning clearing CUDA state: {e}")

try:
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    
    # Freeze CLIP weights
    for param in clip_model.parameters():
        param.requires_grad = False
    
    # Move to device - use cuda:0 explicitly to avoid multi-GPU conflicts
    if torch.cuda.is_available():
        clip_device = torch.device("cuda:0")  # Use first GPU explicitly
        print(f"Moving CLIP model to {clip_device}")
        clip_model = clip_model.to(clip_device)
    else:
        clip_model = clip_model.to(device)
    
    clip_model.eval()
    
    print("✓ CLIP model loaded and frozen!")
    
except RuntimeError as e:
    if "CUDA" in str(e) or "cuda" in str(e).lower():
        print(f"⚠️ CUDA error encountered: {e}")
        print("Attempting to reset CUDA and reload CLIP on CPU...")
        try:
            # Reset CUDA
            torch.cuda.empty_cache()
            # Fallback to CPU
            clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
            clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
            for param in clip_model.parameters():
                param.requires_grad = False
            clip_model = clip_model.to("cpu")
            clip_model.eval()
            print("✓ CLIP model loaded on CPU as fallback")
        except Exception as e2:
            print(f"❌ Failed to load CLIP: {e2}")
            raise
    else:
        raise


Loading CLIP model (ViT-B/32)...
✓ CUDA state cleared


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Moving CLIP model to cuda:0
✓ CLIP model loaded and frozen!


## 3. Load MiniMax-M1-40k

This is the key difference - using MiniMax-M1-40k (456B parameters) for recipe generation!

**Model**: [MiniMaxAI/MiniMax-M1-40k](https://huggingface.co/MiniMaxAI/MiniMax-M1-40k)
- 456B total parameters, 45.9B activated per token
- Supports up to 40K thinking tokens
- Recommended inference: temperature=1.0, top_p=0.95

**⚠️ Memory Requirements**:
- **Full FP16 model**: ~912GB (not feasible on consumer GPUs)
- **With quantization/offloading**: Requires 8-bit quantization + CPU offloading or vLLM
- **Dual RTX 3090 (48GB)**: Will require heavy quantization (4-bit/8-bit) + CPU/disk offloading
- **Recommended**: Use vLLM for production deployment or consider smaller models for local inference


### ⚠️ Memory Requirements & Recommendations

**Reality Check for Dual RTX 3090 (48GB total VRAM)**:

| Configuration | Model Size | Status |
|--------------|------------|--------|
| FP16 (full precision) | ~912GB | ❌ **Impossible** |
| INT8 quantization | ~456GB | ⚠️ **Difficult** - Requires CPU offloading, very slow |
| INT4 quantization | ~228GB | ⚠️ **Possible but slow** - Heavy CPU offloading needed |

**What happens with quantization**:
- ✅ Model will load (using CPU RAM + disk for offloading)
- ⚠️ Inference will be **very slow** (CPU transfers dominate)
- ⚠️ Generation times: potentially minutes per recipe
- ⚠️ May hit system RAM limits (need 200GB+ RAM)

**Better Alternatives**:
1. **Use vLLM** (recommended by model authors) - Better memory management, faster inference
2. **Use API access** - If MiniMax provides hosted API
3. **Use smaller model** - For local development, consider smaller models (7B-70B range)
4. **Cloud deployment** - Deploy on cloud with large GPUs (A100/H100)

**Note**: The MoE architecture helps (only 45.9B activated per token), but loading the full 456B parameter checkpoint still requires massive memory.


### 🔄 Training vs Inference: Important Clarification

**vLLM is for INFERENCE only, not training!**

| Tool | Purpose | Can Train? |
|------|---------|------------|
| **vLLM** | High-speed inference/serving | ❌ **No** - Inference only |
| **Transformers + Trainer** | Training/fine-tuning | ✅ Yes |
| **DeepSpeed** | Large-scale training | ✅ Yes |
| **FSDP (PyTorch)** | Distributed training | ✅ Yes |

**What you can do**:

1. **Use vLLM for inference** (fast generation after model is trained)
   - Deploy the pre-trained MiniMax-M1-40k model
   - Serve it efficiently for recipe generation
   - Best for production serving

2. **Fine-tune with Transformers** (requires different approach)
   - Use Hugging Face `Trainer` or `Trainer` with `accelerate`
   - Requires massive memory even for fine-tuning
   - May need DeepSpeed/FSDP for large models

3. **Training from scratch** (not feasible on consumer hardware)
   - Requires hundreds of GPUs with massive VRAM
   - Typically done on cloud clusters (A100/H100)
   - Cost: Thousands of dollars in compute

**For your dual RTX 3090 setup**:
- ✅ **Fine-tune smaller models** (7B-70B) for recipe generation
- ✅ **Use vLLM** to serve the MiniMax-M1-40k for inference (if you can load it)
- ❌ **Cannot train/fine-tune 456B model** - Insufficient memory


In [ ]:
# Load MiniMax-M1-40k - OPTIMIZED FOR MULTI-GPU WITH QUANTIZATION
print("Loading MiniMax-M1-40k...")
print("This may take a few minutes (model has 456B parameters)...")
print("Model: MiniMaxAI/MiniMax-M1-40k from Hugging Face")

# Check and install required dependencies
try:
    import einops
    print("✓ einops is installed")
except ImportError:
    print("⚠️ einops not found. Installing...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "einops"])
    import einops
    print("✓ einops installed successfully")

# Check available GPU memory
if torch.cuda.is_available():
    total_memory = sum(torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count()))
    total_memory_gb = total_memory / (1024**3)
    print(f"\nAvailable GPU memory: {total_memory_gb:.1f} GB ({torch.cuda.device_count()} GPU(s))")
    print("⚠️ Model requires ~912GB in FP16, will need quantization/offloading")
    
    # Estimate memory needed
    model_size_fp16 = 456 * 2  # 912GB
    model_size_int8 = 456 * 1   # 456GB
    model_size_int4 = 456 * 0.5 # 228GB
    print(f"  - FP16: ~{model_size_fp16}GB (not feasible)")
    print(f"  - INT8: ~{model_size_int8}GB (may work with CPU offloading)")
    print(f"  - INT4: ~{model_size_int4}GB (may fit with offloading)")

# Memory optimization strategy
# For 456B model on dual RTX 3090, we need aggressive quantization
USE_QUANTIZATION = True  # Set to True for memory-constrained systems
QUANTIZATION_BITS = 8    # Options: 4 or 8 (4-bit is more memory efficient but slower)

# Model configuration
model_name = "MiniMaxAI/MiniMax-M1-40k"

try:
    # Load MiniMax-M1-40k model with memory optimizations
    # This is a large hybrid-attention reasoning model (456B params, 45.9B activated per token)
    
    if USE_QUANTIZATION and torch.cuda.is_available():
        print(f"\n📦 Loading with {QUANTIZATION_BITS}-bit quantization + CPU offloading...")
        print("⚠️ This will be slower but will allow the model to fit in memory")
        
        # Configure quantization
        if QUANTIZATION_BITS == 8:
            from transformers import BitsAndBytesConfig
            quantization_config = BitsAndBytesConfig(
                load_in_8bit=True,
                llm_int8_threshold=6.0,
            )
        elif QUANTIZATION_BITS == 4:
            from transformers import BitsAndBytesConfig
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
            )
        else:
            quantization_config = None
        
        minimax_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quantization_config if quantization_config else None,
            device_map="auto",  # Automatically distribute across GPUs/CPU
            low_cpu_mem_usage=True,
            trust_remote_code=True,
            max_memory={i: "20GB" for i in range(torch.cuda.device_count())} if torch.cuda.is_available() else None,
        )
    else:
        # Try loading without quantization (may fail on memory-constrained systems)
        print("\n⚠️ Attempting to load without quantization (may fail if insufficient memory)...")
        print("💡 If this fails, set USE_QUANTIZATION=True above")
        
        minimax_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        )
    
    # Load tokenizer
    minimax_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    
    # Try to load processor for multimodal support (if available)
    try:
        minimax_processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
    except:
        # Fall back to tokenizer if no processor available
        minimax_processor = minimax_tokenizer
        print("⚠️ No processor found, using tokenizer only")
    
    print("✓ MiniMax-M1-40k loaded successfully!")
    print(f"Model parameters: {sum(p.numel() for p in minimax_model.parameters()):,}")
    
    if use_multi_gpu:
        print(f"✓ Model distributed across {num_gpus} GPUs using device_map='auto'")
    
    print("✓ Model supports up to 40K thinking tokens")
    print("✓ Recommended inference params: temperature=1.0, top_p=0.95")
        
except Exception as e:
    print(f"⚠️ Error loading MiniMax-M1-40k: {e}")
    print("\nTroubleshooting:")
    print("  1. Ensure you have enough GPU memory (model is 456B params)")
    print("  2. Check Hugging Face authentication: hf auth login")
    print("  3. Verify model access at https://huggingface.co/MiniMaxAI/MiniMax-M1-40k")
    print("  4. Consider using vLLM for deployment (see model card)")
    import traceback
    traceback.print_exc()
    raise


Loading MiniMax-M1-40k...
This may take a few minutes (model has 456B parameters)...
Model: MiniMaxAI/MiniMax-M1-40k from Hugging Face


A new version of the following files was downloaded from https://huggingface.co/MiniMaxAI/MiniMax-M1-40k:
- configuration_minimax_m1.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
You are using a model of type minimax_m1 to instantiate a model of type MiniMaxM1. This is not supported for all configurations of models and can yield errors.
`torch_dtype` is deprecated! Use `dtype` instead!
Encountered exception while importing einops: No module named 'einops'
Traceback (most recent call last):
  File "/tmp/ipykernel_377645/375230244.py", line 12, in <module>
    minimax_model = AutoModelForCausalLM.from_pretrained(
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/eric/Desktop/CS7643/CLIP+Transformer/venv/lib/python3.12/site-packages/transformers/models/auto/auto_factory.py", line 586, in from_pretrained
    model_class = get_class_from_dynamic_module(
          

⚠️ Error loading MiniMax-M1-40k: This modeling file requires the following packages that were not found in your environment: einops. Run `pip install einops`

Troubleshooting:
  1. Ensure you have enough GPU memory (model is 456B params)
  2. Check Hugging Face authentication: hf auth login
  3. Verify model access at https://huggingface.co/MiniMaxAI/MiniMax-M1-40k
  4. Consider using vLLM for deployment (see model card)


ImportError: This modeling file requires the following packages that were not found in your environment: einops. Run `pip install einops`

## 4. Prepare Dataset

Process the dataset for training with MiniMax-M1-40k.


In [ ]:
# Find image files
def find_image_file(image_name, search_dirs=['food_images/Food Images', 'food_images', '.']):
    extensions = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']
    
    for directory in search_dirs:
        if not os.path.exists(directory):
            continue
        
        for ext in extensions:
            path = os.path.join(directory, image_name + ext)
            if os.path.exists(path) and os.path.isfile(path):
                return path
    
    return None

# Process dataset for MiniMax-M1-40k
processed_data = []

for idx, row in main_dataset.iterrows():
    image_name = row['Image_Name']
    image_path = find_image_file(image_name)
    
    if image_path is None:
        continue
    
    # Format recipe for MiniMax-M1-40k
    title = row['Title']
    ingredients = row['Ingredients']
    instructions = row['Instructions']
    
    # Create structured output
    recipe_output = f"""RECIPE: {title}

INGREDIENTS:
{ingredients}

INSTRUCTIONS:
{instructions}"""
    
    processed_data.append({
        'image_path': image_path,
        'title': title,
        'recipe': recipe_output
    })

print(f"✓ Processed {len(processed_data)} samples out of {len(main_dataset)} total")
if processed_data:
    print(f"✓ First sample: {processed_data[0]['title']}")


## 5. Recipe Generation Function

Function to generate recipes using CLIP + MiniMax-M1-40k.

The MiniMax-M1-40k model leverages extended reasoning capabilities (up to 40K thinking tokens) to generate detailed, coherent recipes with better instruction following.


In [ ]:
def recognize_food_with_clip(image, clip_model, clip_processor):
    """
    Use CLIP to recognize food category
    """
    food_categories = [
        "grilled chicken", "roasted chicken", "fried chicken",
        "beef steak", "grilled beef", "salmon", "grilled salmon",
        "pasta", "spaghetti", "pizza", "burger", "salad",
        "soup", "sandwich", "cake", "pie", "cookies",
        "sushi", "tacos", "curry", "stir fry"
    ]
    
    try:
        # Get the device CLIP model is on
        clip_device = next(clip_model.parameters()).device
        
        text_inputs = [f"a photo of {food}" for food in food_categories]
        inputs = clip_processor(text=text_inputs, images=image, return_tensors="pt", padding=True)
        inputs = {k: v.to(clip_device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = clip_model(**inputs)
            logits_per_image = outputs.logits_per_image
            probs = logits_per_image.softmax(dim=1)
        
        top_idx = probs[0].argmax().item()
        return food_categories[top_idx]
    except Exception as e:
        print(f"⚠️ CLIP recognition error (using generic): {str(e)[:50]}")
        return "food dish"


def generate_recipe_with_minimax(image_path, minimax_model, minimax_tokenizer, clip_model, clip_processor, max_new_tokens=1024, skip_clip=False):
    """
    Generate recipe using CLIP + MiniMax-M1-40k
    
    Args:
        image_path: Path to food image
        minimax_model: MiniMax-M1-40k model
        minimax_tokenizer: MiniMax-M1-40k tokenizer
        clip_model: CLIP model for food recognition
        clip_processor: CLIP processor
        max_new_tokens: Maximum tokens to generate (can be up to 40K for thinking tokens)
        skip_clip: Skip CLIP recognition for faster generation
    """
    try:
        # Load image
        image = Image.open(image_path).convert('RGB')
        
        # Recognize food with CLIP first (skip for speed if requested)
        if skip_clip:
            food_type = "food dish"
        else:
            food_type = recognize_food_with_clip(image, clip_model, clip_processor)
        
        # Create prompt for MiniMax-M1-40k
        prompt_text = f"""You are an expert chef. Generate a detailed recipe for this {food_type}.

Include:
1. RECIPE TITLE
2. INGREDIENTS LIST with quantities and measurements
3. STEP-BY-STEP INSTRUCTIONS

Begin:"""
        
        # Tokenize input
        device = next(minimax_model.parameters()).device
        inputs = minimax_tokenizer(prompt_text, return_tensors="pt").to(device)
        
        # Generate with MiniMax-M1-40k
        # Using recommended inference parameters: temperature=1.0, top_p=0.95
        minimax_model.eval()
        
        with torch.inference_mode():
            generated_ids = minimax_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=1.0,  # Recommended parameter from model card
                top_p=0.95,       # Recommended parameter from model card
                pad_token_id=minimax_tokenizer.eos_token_id if minimax_tokenizer.eos_token_id is not None else minimax_tokenizer.pad_token_id
            )
        
        # Decode
        generated_text = minimax_tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        
        # Extract only the generated part (excluding prompt)
        recipe = generated_text[len(prompt_text):].strip()
        
        return {
            'recognized_food': food_type,
            'recipe': recipe,
            'image': image
        }
        
    except Exception as e:
        import traceback
        traceback.print_exc()
        return {
            'recognized_food': 'unknown',
            'recipe': f"Error generating recipe: {str(e)}",
            'image': None
        }

print("✓ Recipe generation functions defined!")


In [ ]:
# Test on a few samples
if processed_data:
    print("\nTesting recipe generation...")
    print("=" * 60)
    
    test_samples = random.sample(processed_data, min(3, len(processed_data)))
    
    for i, sample in enumerate(test_samples, 1):
        print(f"\nTest {i}/{len(test_samples)}")
        print("-" * 60)
        print(f"Original: {sample['title']}")
        
        result = generate_recipe_with_minimax(
            sample['image_path'],
            minimax_model,
            minimax_tokenizer,
            clip_model,
            clip_processor
        )
        
        print(f"Recognized as: {result['recognized_food']}")
        print(f"\nGenerated Recipe:\n{result['recipe'][:500]}...")
        
        # Display image
        if result['image']:
            plt.figure(figsize=(8, 6))
            plt.imshow(result['image'])
            plt.title(f"Recognized: {result['recognized_food']}")
            plt.axis('off')
            plt.show()
    
    print("\n" + "=" * 60)
    print("✓ Testing complete!")


## 7. Create Gradio Web Interface

Interactive web interface powered by CLIP + MiniMax-M1-40k for recipe generation from food images.


In [ ]:
import gradio as gr

def gradio_generate_recipe(image):
    """
    Wrapper function for Gradio interface
    """
    if image is None:
        return "⚠️ Please upload an image."
    
    try:
        # Save temp image
        temp_path = "temp_food_image.jpg"
        if isinstance(image, np.ndarray):
            Image.fromarray(image).save(temp_path)
        else:
            image.save(temp_path)
        
        # Generate recipe
        result = generate_recipe_with_minimax(
            temp_path,
            minimax_model,
            minimax_tokenizer,
            clip_model,
            clip_processor,
            max_new_tokens=1024,
            skip_clip=False
        )
        
        # Format output
        output = f"""RECOGNIZED FOOD: {result['recognized_food'].upper()}

{'=' * 60}

{result['recipe']}

{'=' * 60}
NOTE: This recipe was generated by AI and may require refinement.
Please verify ingredients and instructions before cooking.
"""
        
        # Cleanup
        if os.path.exists(temp_path):
            os.remove(temp_path)
        
        return output
        
    except Exception as e:
        return f"❌ Error: {str(e)}"


# Create Gradio interface
with gr.Blocks(title="Food Recipe Generator (MiniMax-M1-40k)", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🍽️ Food to Recipe Generator
    ### Powered by CLIP + MiniMax-M1-40k
    
    Upload a food image and get a detailed AI-generated recipe with:
    - 📋 Complete ingredient list
    - ⏱️ Step-by-step instructions
    - 🍳 Cooking guidance
    - 🧠 Extended reasoning (40K thinking tokens)
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(
                label="Upload Food Image",
                type="pil",
                height=400
            )
            generate_btn = gr.Button(
                "🚀 Generate Recipe",
                variant="primary",
                size="lg"
            )
            
            gr.Markdown("""
            ### Tips:
            - Upload clear food images
            - Works best with single dishes
            - Powered by state-of-the-art MiniMax-M1-40k
            - Supports extended reasoning for detailed recipes
            """)
        
        with gr.Column(scale=1):
            recipe_output = gr.Textbox(
                label="Generated Recipe",
                lines=25,
                max_lines=30,
                placeholder="Your recipe will appear here...",
                show_copy_button=True
            )
    
    # Connect function
    generate_btn.click(
        fn=gradio_generate_recipe,
        inputs=image_input,
        outputs=recipe_output
    )
    
    image_input.upload(
        fn=gradio_generate_recipe,
        inputs=image_input,
        outputs=recipe_output
    )
    
    gr.Markdown("""
    ---
    **Note**: Recipes are AI-generated and may need refinement.
    
    Built with [CLIP](https://openai.com/research/clip) + [MiniMax-M1-40k](https://huggingface.co/MiniMaxAI/MiniMax-M1-40k) + [Gradio](https://gradio.app)
    """)

print("✓ Gradio interface created!")


## 8. Launch Web Interface

Launch the Gradio interface to interact with the MiniMax-M1-40k recipe generator.


## 9. Training/Fine-tuning Considerations

### Why You Can't Train/Fine-tune 456B Model Locally

**Memory Requirements for Training**:
- **Inference**: Model weights + activations (~912GB FP16)
- **Training**: Model weights + activations + gradients + optimizer states
  - **Gradients**: ~912GB (same as model)
  - **Adam optimizer states**: ~1.8TB (2x model for momentum + variance)
  - **Total training memory**: ~3.6TB+ for full precision training

**Your Hardware**: Dual RTX 3090 = 48GB VRAM
- ❌ Cannot hold even inference (912GB needed)
- ❌ Cannot train/fine-tune (3.6TB+ needed)

### Practical Alternatives for Training

#### Option 1: Fine-tune Smaller Models (Recommended)
Train a smaller model that fits your hardware:
- **Llama 3 8B/70B** - Popular choice, good recipe generation
- **Mistral 7B** - Efficient, fits on single RTX 3090
- **Phi-3** - Microsoft's efficient models
- **Fine-tuning approach**: Use LoRA/QLoRA to reduce memory

#### Option 2: Use vLLM for Inference Only
- Load pre-trained MiniMax-M1-40k via vLLM
- Use it for inference (recipe generation)
- Cannot modify/fine-tune the model

#### Option 3: Cloud Training
- Use cloud services (AWS, GCP, Azure) with A100/H100 GPUs
- Use DeepSpeed/FSDP for distributed training
- Cost: $10-100/hour depending on cluster size

### Code Structure for Fine-tuning (Smaller Models)

```python
# Example for smaller models (7B-70B)
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=1,  # Very small for large models
    gradient_accumulation_steps=32,  # Accumulate gradients
    gradient_checkpointing=True,     # Save memory
    fp16=True,                        # Mixed precision
    logging_steps=10,
    save_steps=1000,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

trainer.train()
```

**For 456B model**: Even with all optimizations, requires 500GB+ VRAM for fine-tuning.


In [ ]:
# Launch the interface
try:
    demo.close()
except:
    pass

demo.launch(
    share=True,  # Set to True for public link
    server_name="0.0.0.0",
    server_port=None,
    show_error=True,
    inbrowser=True
)


## Summary

### Key Features:

1. **Model**: MiniMax-M1-40k (456B parameters, 40K thinking tokens) for recipe generation
2. **Architecture**: Uses CLIP for vision understanding + MiniMax-M1-40k for text generation
3. **Approach**: CLIP recognizes food type, then MiniMax-M1-40k generates detailed recipe with extended reasoning
4. **Quality**: Large-scale hybrid-attention model for better coherence and accuracy

### Advantages:
- ✅ Large-scale hybrid-attention reasoning model
- ✅ Extended thinking capability (40K tokens)
- ✅ Better recipe quality with reinforcement learning training
- ✅ Clearer instructions and better instruction following
- ✅ Integration with CLIP for better understanding

### Model Details:
- **Model**: [MiniMaxAI/MiniMax-M1-40k](https://huggingface.co/MiniMaxAI/MiniMax-M1-40k)
- **Parameters**: 456B total, 45.9B activated per token
- **Thinking Tokens**: Up to 40K
- **Recommended Inference**: temperature=1.0, top_p=0.95
- **Context Length**: 1 million tokens

### Notes:
- ⚠️ Model requires significant GPU memory (456B parameters)
- ⚠️ Consider using vLLM for production deployment (see model card)
- ⚠️ Ensure Hugging Face authentication: `hf auth login`

### Training Limitations:
- ❌ **Cannot fine-tune 456B model** on dual RTX 3090 (requires 3.6TB+ memory)
- ✅ **Can use for inference** with quantization/offloading (slow but possible)
- ✅ **Can fine-tune smaller models** (7B-70B) for recipe-specific tasks
- ✅ **Use vLLM** for efficient inference serving (recommended by authors)

### Practical Next Steps:
1. **For inference**: Use vLLM to serve MiniMax-M1-40k (best performance)
2. **For fine-tuning**: Train a smaller model (7B-13B) on your recipe dataset
3. **Hybrid approach**: Use MiniMax-M1-40k API (if available) + fine-tuned smaller model

**Overall**: MiniMax-M1-40k provides state-of-the-art reasoning capabilities for recipe generation, but requires specialized infrastructure for training/fine-tuning. For local development, consider smaller models or cloud deployment.
